# 02 -- Bias Analysis: Model Fitting & Noise Sensitivity

**Corresponds to:** Manuscript Sec.2.2 (Model Validation), Supplemental Tables & Figures

This notebook combines two complementary bias analyses:

**Part A — Model Fitting Bias:** Runs the IVIM multi-compartment model fitting pipeline on synthetic data with known ground truth parameters across varied tissue compositions (GM/WM at tissue fractions of 0.5 and 0.85). Quantifies bias in estimated volume fractions and DTI/DKI metrics.

**Part B — Noise-Induced Bias:** Loads multi-SNR simulation results from `01_noise_testing.ipynb` and characterizes how Rician noise (5--80 dB) affects estimation accuracy.

**Key outputs:**
- Combined 2×2 grid of bias box plots for all tissue × tissue-type conditions
- Table of percent bias statistics at specific SNR levels
- Box plots, heatmaps, and line plots of bias vs SNR
- Summary statistics table

## Setup & Imports

In [ ]:
from dmipy.signal_models import cylinder_models, gaussian_models
from dmipy.core import modeling_framework
from dmipy.core.modeling_framework import MultiCompartmentModel
from dmipy.signal_models.cylinder_models import C2CylinderStejskalTannerApproximation
from dmipy.core.acquisition_scheme import acquisition_scheme_from_bvalues, gtab_dmipy2dipy
from dipy.io import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.dti import TensorModel
import dipy.reconst.dki as dki
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
np.random.seed(41)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10
print("Setup complete.")

---
# Part A: Model Fitting Bias (Composition Sweep)
---

We sweep over ground-truth configurations with varying free-water (fw) and perfusion (fb) fractions at fixed tissue fractions. At each configuration:
1. Generate dummy multi-compartment signal using Dmipy
2. Fit the IVIM model with fixed lambda_iso parameters
3. Reconstruct cylinder-only signal using fitted orientation/diameter
4. Compute DTI (FA, MD, RD, AD) and DKI (KFA, MK, RK, AK) from the cylinder signal
5. Compute ground-truth DTI/DKI metrics from the known cylinder geometry
6. Save results to CSV for downstream visualization

### A.1 Configure Paths & Acquisition Scheme

In [ ]:
input_directory = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\input_data\WM"
output_directory = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\output_data\WM"

fbval = input_directory + "/ivimbvals.txt"
fbvec = input_directory + "/ivimbvecs.txt"

bvals, bvecs = read_bvals_bvecs(fbval, fbvec)
gtab = gradient_table(bvals, bvecs)
bvalues_SI = bvals * 1e6  # SI units (s/m^2)

# Acquisition scheme
delta = 0.0106
Delta = 0.0431
scheme_ivim = acquisition_scheme_from_bvalues(
    bvalues_SI, bvecs, delta, Delta, b0_threshold=1e6, min_b_shell_distance=1e7
)
print(f"Acquisition scheme: {scheme_ivim.nmr_echo_parameters.shape[0]} measurements")

### A.2 Helper Functions

In [ ]:
def generate_dummy_parameters(tissue, fw_min, fw_max, n):
    fw = np.linspace(fw_min, fw_max, n).tolist()
    fb = np.linspace(1 - tissue - fw_min, 1 - tissue - fw_max, n).tolist()
    return fw, fb


def fit_dti(scheme, signal):
    gtab_dipy = gtab_dmipy2dipy(scheme)
    tenmod = TensorModel(gtab_dipy)
    tenfit = tenmod.fit(signal)
    return {
        'fa': np.mean(tenfit.fa),
        'md': np.mean(tenfit.md),
        'rd': np.mean(tenfit.rd),
        'ad': np.mean(tenfit.ad)
    }


def fit_dki(scheme, signal):
    gtab_dipy = gtab_dmipy2dipy(scheme)
    dkimodel = dki.DiffusionKurtosisModel(gtab_dipy)
    dkifit = dkimodel.fit(signal)
    return {
        'kfa': np.mean(dkifit.kfa),
        'mk': np.mean(dkifit.mk()),
        'rk': np.mean(dkifit.rk()),
        'ak': np.mean(dkifit.ak())
    }


def print_metrics(label, metrics):
    print(f"\n--- {label} ---")
    for k, v in metrics.items():
        print(f"{k.upper()}: {v:.6f}")

### A.3 Run Fitting Sweep

We sweep over `n = 30` ground-truth configurations at tissue fraction 0.5.

In [ ]:
n = 30
tissue = 0.5
fw, fb = generate_dummy_parameters(tissue, 0.4, 0.49, n)
print(f"FB range: {min(fb):.4f} -- {max(fb):.4f}")
print(f"FW range: {min(fw):.4f} -- {max(fw):.4f}")

total_data = []

for i in tqdm(range(n)):
    row = {}

    # Define IVIM-FWI model
    ball1 = gaussian_models.G1Ball()
    ball2 = gaussian_models.G1Ball()
    cyl = cylinder_models.C2CylinderStejskalTannerApproximation()
    ballcyl = modeling_framework.MultiCompartmentModel([ball1, ball2, cyl])

    # Dummy parameters
    params = {
        'G1Ball_1_lambda_iso': 3e-9,
        'G1Ball_2_lambda_iso': 7e-9,
        'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9,
        'C2CylinderStejskalTannerApproximation_1_diameter': 1e-6,
        'C2CylinderStejskalTannerApproximation_1_mu': [0.4, 0.4],
        'partial_volume_0': fw[i],
        'partial_volume_1': fb[i],
        'partial_volume_2': tissue
    }
    dummy_data = ballcyl(scheme_ivim, **params)

    # Fit IVIM model
    ivim_mod = MultiCompartmentModel([ball1, ball2, C2CylinderStejskalTannerApproximation()])
    ivim_mod.set_fixed_parameter('G1Ball_1_lambda_iso', 3e-9)
    ivim_mod.set_fixed_parameter('G1Ball_2_lambda_iso', 7e-9)
    ivim_mod.set_fixed_parameter('C2CylinderStejskalTannerApproximation_1_lambda_par', 1.7e-9)

    ivim_fit = ivim_mod.fit(acquisition_scheme=scheme_ivim, data=dummy_data)
    fitted_params = ivim_fit.fitted_parameters

    row['fw_val'] = fw[i]
    row['fb_val'] = fb[i]
    row['fitted_fw'] = np.mean(fitted_params['partial_volume_0'])
    row['fitted_fb'] = np.mean(fitted_params['partial_volume_1'])
    row['tissue_frac'] = np.mean(fitted_params['partial_volume_2'])

    # Cylinder-only signal
    cyl_only_model = modeling_framework.MultiCompartmentModel([cyl])
    cyl_params = {
        'C2CylinderStejskalTannerApproximation_1_mu': fitted_params['C2CylinderStejskalTannerApproximation_1_mu'],
        'C2CylinderStejskalTannerApproximation_1_diameter': fitted_params['C2CylinderStejskalTannerApproximation_1_diameter'],
        'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9
    }
    signal_cyl_only = cyl_only_model.simulate_signal(scheme_ivim, cyl_params)

    # DTI metrics
    dti_metrics = fit_dti(scheme_ivim, signal_cyl_only)
    row.update({f'dti_mean_{k}': v for k, v in dti_metrics.items()})

    # DKI metrics
    dki_metrics = fit_dki(scheme_ivim, signal_cyl_only)
    row.update({f'dkifit_mean_{k}': v for k, v in dki_metrics.items()})

    # Ground truth metrics (fixed cylinder)
    gt_params = {
        'C2CylinderStejskalTannerApproximation_1_mu': [0.4, 0.4],
        'C2CylinderStejskalTannerApproximation_1_diameter': 1e-6,
        'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9
    }
    gt_signal = cyl_only_model.simulate_signal(scheme_ivim, gt_params)
    dummy_cyldata = np.tile(gt_signal, [10, 1])

    gt_dti_metrics = fit_dti(scheme_ivim, dummy_cyldata)
    row.update({f'ground_truth_dti_{k}': v for k, v in gt_dti_metrics.items()})

    gt_dki_metrics = fit_dki(scheme_ivim, dummy_cyldata)
    row.update({f'ground_truth_dkifit_{k}': v for k, v in gt_dki_metrics.items()})

    total_data.append(row)

os.makedirs(output_directory, exist_ok=True)
df_fitting = pd.DataFrame(total_data)
df_fitting['tissue_input'] = tissue

# Save intermediate results
output_csv = os.path.join(output_directory, f'output-fb-{tissue:.2f}-wm.csv')
df_fitting.to_csv(output_csv, index=False)
print(f"Results saved to: {output_csv}")
print(f"Completed {n} iterations.")

### A.4 Compute Bias Metrics

For each dataset, compute relative bias for volume fractions and diffusion metrics: \\( \\text{Bias} = \\frac{\\text{Fitted} - \\text{True}}{\\text{True}} \\).

In [ ]:
def compute_biases(df):
    """Compute all bias metrics for a dataframe."""
    df['bias_fw'] = (df['fitted_fw'] - df['fw_val']) / df['fw_val']
    df['bias_fb'] = (df['fitted_fb'] - df['fb_val']) / df['fb_val']
    df['bias_tissue'] = (df['tissue_frac'] - df['tissue_input']) / df['tissue_input']
    df['bias_FA'] = (df['dti_mean_fa'] - df['ground_truth_dti_fa']) / df['ground_truth_dti_fa']
    df['bias_MD'] = (df['dti_mean_md'] - df['ground_truth_dti_md']) / df['ground_truth_dti_md']
    df['bias_KFA'] = (df['dkifit_mean_kfa'] - df['ground_truth_dkifit_kfa']) / df['ground_truth_dkifit_kfa']
    df['bias_MK'] = (df['dkifit_mean_mk'] - df['ground_truth_dkifit_mk']) / df['ground_truth_dkifit_mk']
    return {
        "FW": df['bias_fw'], "FB": df['bias_fb'],
        "FA": df['bias_FA'], "MD": df['bias_MD'],
        "KFA": df['bias_KFA'], "MK": df['bias_MK'],
    }


metrics = compute_biases(df_fitting)
print("Bias Means and Standard Deviations (tissue=0.5, WM):")
for m, series in metrics.items():
    print(f"{m}: mean={series.mean():.4f}, std={series.std():.4f}")

### A.5 FB Bias vs Input FB

Plot the bias in the perfusion fraction estimate as a function of the ground-truth FB fraction.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(df_fitting['fb_val'], df_fitting['bias_fb'], marker='o', color='royalblue', label=f"Tissue={tissue:.2f}")
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel("Input FB fraction")
plt.ylabel("Bias = (Fitted - Actual) / Actual")
plt.title(f"FB Bias at Tissue Fraction {tissue:.2f} in WM")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### A.6 Combined Bias Box Plots (All Conditions)

Load CSV outputs across four conditions (GM and WM at tissue fractions of 0.5 and 0.85) and generate a combined 2×2 grid:
- **Top left:** Tissue fraction 0.5, Gray Matter
- **Top right:** Tissue fraction 0.5, White Matter
- **Bottom left:** Tissue fraction 0.85, Gray Matter
- **Bottom right:** Tissue fraction 0.85, White Matter

Within each panel, a 3×2 subgrid shows the distribution of bias for each metric (FW, FB, FA, MD, KFA, MK).

In [ ]:
def create_combined_plot(files, output_png):
    """Create a 2x2 grid of plots for all four conditions."""

    labels = {
        files[0]: ("0.5 GM", 0, 0),
        files[1]: ("0.5 WM", 0, 1),
        files[2]: ("0.85 GM", 1, 0),
        files[3]: ("0.85 WM", 1, 1),
    }

    fig = plt.figure(figsize=(16, 12))

    for file_idx, csv_file in enumerate(files):
        print(f"\nProcessing: {csv_file}")
        df = pd.read_csv(csv_file)
        metrics = compute_biases(df)

        print("Bias Means and Standard Deviations:")
        for m, series in metrics.items():
            print(f"{m}: mean={series.mean():.4f}, std={series.std():.4f}")

        label, row, col = labels[csv_file]

        for metric_idx, (name, series) in enumerate(metrics.items()):
            subplot_row = row * 3 + (metric_idx // 2)
            subplot_col = col * 2 + (metric_idx % 2)

            ax = plt.subplot2grid((6, 4), (subplot_row, subplot_col))
            ax.boxplot(series, vert=False)
            ax.set_title(f"{name}", fontsize=10)
            ax.set_xlabel("Bias", fontsize=9)
            ax.set_yticks([])
            ax.locator_params(axis='x', nbins=5)
            ax.tick_params(labelsize=8)

    # Row and column labels
    fig.text(0.02, 0.75, '0.5', fontsize=16, fontweight='bold',
             rotation=90, va='center', ha='center')
    fig.text(0.02, 0.28, '0.85', fontsize=16, fontweight='bold',
             rotation=90, va='center', ha='center')
    fig.text(0.27, 0.02, 'GM', fontsize=16, fontweight='bold',
             ha='center', va='center')
    fig.text(0.77, 0.02, 'WM', fontsize=16, fontweight='bold',
             ha='center', va='center')

    plt.tight_layout(rect=[0.03, 0.03, 1, 0.98], h_pad=5, w_pad=5)
    plt.subplots_adjust(hspace=0.6, wspace=0.3)
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    print(f"\nSaved combined plot to: {output_png}")
    plt.close()


# Process all files in a single combined plot
bias_files = [
    "output_data/full_brain_1/output-fb-0.50-gm.csv",
    "output_data/full_brain_1/output-fb-0.50-wm.csv",
    "output_data/full_brain_1/output-fb-0.85-gm.csv",
    "output_data/full_brain_1/output-fb-0.85-wm.csv"
]

existing_bias = [f for f in bias_files if os.path.exists(f)]
print(f"Found {len(existing_bias)}/{len(bias_files)} input files")

if len(existing_bias) == 4:
    output_png = "output_data/full_brain_1/combined_bias_plots_2x2.png"
    os.makedirs(os.path.dirname(output_png), exist_ok=True)
    create_combined_plot(bias_files, output_png)
    print("\nCombined bias plot saved successfully!")
else:
    print("\nSkipping combined plot: not all input files found.")
    print("Generate the missing CSVs by running this notebook's sweep for each condition.")

---
# Part B: Noise-Induced Bias Analysis
---

Load the multi-SNR simulation output from `01_noise_testing.ipynb` and characterize how noise affects estimation accuracy. Ground truth values: fw = 0.10, pf = 0.05, tissue = 0.85.

### B.1 Load Noise Simulation Data

In [ ]:
INPUT_CSV = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\output_data\full_brain_noise\ivim_output.csv"
OUTPUT_DIR = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\output_data\full_brain_noise\figures"

# Ground truth values
GROUND_TRUTH = {'fw': 0.10, 'pf': 0.05, 'tissue': 0.85}
SNR_LEVELS = [5, 20, 40, 60, 80]

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if os.path.exists(INPUT_CSV):
    df_noise = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df_noise)} SNR configurations")
    print(f"SNR range: {df_noise['SNR'].min():.1f} -- {df_noise['SNR'].max():.1f} dB")
    DATA_LOADED = True
else:
    print(f"Input file not found: {INPUT_CSV}")
    print("Run 01_noise_testing.ipynb first.")
    DATA_LOADED = False

### B.2 Calculate Percent Bias

In [ ]:
def calculate_percent_bias(estimated, ground_truth):
    return ((estimated - ground_truth) / ground_truth) * 100

if DATA_LOADED:
    df_noise['fw_percent_bias'] = calculate_percent_bias(df_noise['fw_mean'], GROUND_TRUTH['fw'])
    df_noise['pf_percent_bias'] = calculate_percent_bias(df_noise['pf_mean'], GROUND_TRUTH['pf'])
    df_noise['tissue_percent_bias'] = calculate_percent_bias(df_noise['tissue_mean'], GROUND_TRUTH['tissue'])
    df_noise['fa_percent_bias'] = calculate_percent_bias(df_noise['fitted_fa_mean'], df_noise['gt_fa_mean'])
    df_noise['md_percent_bias'] = calculate_percent_bias(df_noise['fitted_md_mean'], df_noise['gt_md_mean'])
    print("Percent bias columns added.")

### B.3 Table 2: Statistics at Specific SNR Levels

In [ ]:
def generate_table2():
    """Generate Table 2 with statistics at specific SNR levels."""
    print("\n" + "=" * 80)
    print("TABLE 2: Percent Bias of Estimated IVIM Metrics vs Ground Truth")
    print("=" * 80)

    table_data = []
    for snr_target in SNR_LEVELS:
        idx = (df_noise['SNR'] - snr_target).abs().idxmin()
        row = df_noise.loc[idx]

        stats = {
            'SNR': f"{row['SNR']:.1f}",
            'fw_bias': row['fw_percent_bias'],
            'fw_std': (row['fw_std'] / GROUND_TRUTH['fw']) * 100,
            'pf_bias': row['pf_percent_bias'],
            'pf_std': (row['pf_std'] / GROUND_TRUTH['pf']) * 100,
            'tissue_bias': row['tissue_percent_bias'],
            'tissue_std': (row['tissue_std'] / GROUND_TRUTH['tissue']) * 100,
            'fa_bias': row['fa_percent_bias'],
            'fa_std': (row['fitted_fa_std'] / row['fitted_fa_mean']) * 100 if row['fitted_fa_mean'] > 0 else 0,
            'md_bias': row['md_percent_bias'],
            'md_std': (row['fitted_md_std'] / row['fitted_md_mean']) * 100 if row['fitted_md_mean'] > 0 else 0
        }
        table_data.append(stats)

        print(f"\nSNR = {row['SNR']:.1f} dB")
        print(f"  Free Water (fw):    Bias = {stats['fw_bias']:>6.2f}% \u00b1 {stats['fw_std']:>6.2f}%")
        print(f"  Perfusion (pf):     Bias = {stats['pf_bias']:>6.2f}% \u00b1 {stats['pf_std']:>6.2f}%")
        print(f"  Tissue (ft):        Bias = {stats['tissue_bias']:>6.2f}% \u00b1 {stats['tissue_std']:>6.2f}%")
        print(f"  FA:                 Bias = {stats['fa_bias']:>6.2f}% \u00b1 {stats['fa_std']:>6.2f}%")
        print(f"  MD:                 Bias = {stats['md_bias']:>6.2f}% \u00b1 {stats['md_std']:>6.2f}%")

    table_df = pd.DataFrame(table_data)
    table_df.to_csv(Path(OUTPUT_DIR) / "table2_percent_bias.csv", index=False)
    print(f"\nTable saved to: {Path(OUTPUT_DIR) / 'table2_percent_bias.csv'}")
    return table_df


if DATA_LOADED:
    table2_df = generate_table2()

### B.4 Box Plots of Percent Bias by Metric

In [ ]:
def plot_noise_boxplots():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Percent Bias Distribution Across SNR Levels',
                 fontsize=14, fontweight='bold', y=0.995)

    noise_metrics = [
        ('fw_percent_bias', 'Free Water (fw)', axes[0, 0]),
        ('pf_percent_bias', 'Perfusion Fraction (pf)', axes[0, 1]),
        ('tissue_percent_bias', 'Tissue Fraction (ft)', axes[0, 2]),
        ('fa_percent_bias', 'Fractional Anisotropy (FA)', axes[1, 0]),
        ('md_percent_bias', 'Mean Diffusivity (MD)', axes[1, 1])
    ]

    df_noise['SNR_bin'] = pd.cut(df_noise['SNR'], bins=5,
                                 labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

    for metric, title, ax in noise_metrics:
        sns.boxplot(data=df_noise, x='SNR_bin', y=metric, ax=ax, palette='viridis')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('SNR Level')
        ax.set_ylabel('Percent Bias (%)')
        ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Ground Truth')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.tick_params(axis='x', rotation=45)

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figure2_boxplots.png", dpi=300, bbox_inches='tight')
    print(f"Box plots saved to: {Path(OUTPUT_DIR) / 'figure2_boxplots.png'}")
    plt.close()


if DATA_LOADED:
    plot_noise_boxplots()

### B.5 Heatmaps of Percent Change vs SNR

In [ ]:
def plot_heatmaps():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Percent Bias vs SNR (Heatmap View)',
                 fontsize=14, fontweight='bold', y=0.995)

    snr_sorted = df_noise.sort_values('SNR')
    n_points = len(snr_sorted)

    hmap_metrics = [
        ('fw_percent_bias', 'Free Water (fw)'),
        ('pf_percent_bias', 'Perfusion Fraction (pf)'),
        ('tissue_percent_bias', 'Tissue Fraction (ft)'),
        ('fa_percent_bias', 'Fractional Anisotropy (FA)'),
        ('md_percent_bias', 'Mean Diffusivity (MD)')
    ]

    for idx, (metric, title) in enumerate(hmap_metrics):
        ax = axes[idx // 3, idx % 3]
        grid_size = int(np.sqrt(n_points))
        data_grid = snr_sorted[metric].values[:grid_size ** 2].reshape(grid_size, grid_size)

        im = ax.imshow(data_grid, cmap='RdBu_r', aspect='auto', vmin=-50, vmax=50)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Sample Index')
        ax.set_ylabel('Sample Index')
        plt.colorbar(im, ax=ax, label='Percent Bias (%)')

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figure3_heatmaps.png", dpi=300, bbox_inches='tight')
    print(f"Heatmaps saved to: {Path(OUTPUT_DIR) / 'figure3_heatmaps.png'}")
    plt.close()


if DATA_LOADED:
    plot_heatmaps()

### B.6 Line Plots — Bias vs SNR

Mean percent bias with shaded \u00b11 SD regions as a function of SNR.

In [ ]:
def plot_bias_vs_snr():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Percent Bias vs Signal-to-Noise Ratio',
                 fontsize=14, fontweight='bold', y=0.995)

    line_metrics = [
        ('fw_percent_bias', 'fw_std', 'Free Water (fw)', 'blue'),
        ('pf_percent_bias', 'pf_std', 'Perfusion Fraction (pf)', 'green'),
        ('tissue_percent_bias', 'tissue_std', 'Tissue Fraction (ft)', 'orange'),
        ('fa_percent_bias', 'fitted_fa_std', 'Fractional Anisotropy (FA)', 'red'),
        ('md_percent_bias', 'fitted_md_std', 'Mean Diffusivity (MD)', 'purple')
    ]

    df_sorted = df_noise.sort_values('SNR')

    for idx, (bias_col, std_col, title, color) in enumerate(line_metrics):
        ax = axes[idx // 3, idx % 3]
        bias_values = df_sorted[bias_col].values
        std_values = df_sorted[std_col].values

        ax.plot(df_sorted['SNR'], bias_values, linewidth=2, color=color, label='Mean Bias')
        ax.fill_between(df_sorted['SNR'],
                        bias_values - std_values,
                        bias_values + std_values,
                        alpha=0.3, color=color, label=' \u00b11 SD')

        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel('SNR (dB)')
        ax.set_ylabel('Percent Bias (%)')
        ax.set_title(title, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figure4_bias_vs_snr.png", dpi=300, bbox_inches='tight')
    print(f"Line plots saved to: {Path(OUTPUT_DIR) / 'figure4_bias_vs_snr.png'}")
    plt.close()


if DATA_LOADED:
    plot_bias_vs_snr()

### B.7 Summary Statistics Table

Aggregate percent bias statistics across all SNR levels.

In [ ]:
def plot_summary_table():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axis('tight')
    ax.axis('off')

    summary_stats = []
    for metric in ['fw', 'pf', 'tissue']:
        bias_col = f'{metric}_percent_bias'
        summary_stats.append([metric.upper(),
                              f"{df_noise[bias_col].mean():.2f}%",
                              f"{df_noise[bias_col].std():.2f}%",
                              f"{df_noise[bias_col].min():.2f}%",
                              f"{df_noise[bias_col].max():.2f}%"])

    for metric in ['fa', 'md']:
        bias_col = f'{metric}_percent_bias'
        summary_stats.append([metric.upper(),
                              f"{df_noise[bias_col].mean():.2f}%",
                              f"{df_noise[bias_col].std():.2f}%",
                              f"{df_noise[bias_col].min():.2f}%",
                              f"{df_noise[bias_col].max():.2f}%"])

    table = ax.table(cellText=summary_stats,
                     colLabels=['Metric', 'Mean Bias', 'SD Bias', 'Min Bias', 'Max Bias'],
                     cellLoc='center', loc='center',
                     colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    for i in range(5):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')

    plt.title('Summary Statistics: Percent Bias Across All SNR Levels',
              fontweight='bold', fontsize=12, pad=20)
    plt.savefig(Path(OUTPUT_DIR) / "figure5_summary_table.png", dpi=300, bbox_inches='tight')
    print(f"Summary table saved to: {Path(OUTPUT_DIR) / 'figure5_summary_table.png'}")
    plt.close()


if DATA_LOADED:
    plot_summary_table()

### B.8 Export Full Analysis Dataset

In [ ]:
if DATA_LOADED:
    output_full = Path(OUTPUT_DIR) / "full_analysis_with_bias.csv"
    df_noise.to_csv(output_full, index=False)
    print(f"Full analysis dataset saved to: {output_full}")

print("\n" + "=" * 80)
print("BIAS ANALYSIS COMPLETE")
print("=" * 80)